In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy import units as u
from astropy.coordinates import Galactic
from astropy.coordinates import SkyCoord

In [ ]:
def FD_moments(data,peakthresh,maxFD,RM_arr,M0=True,M1=True,M2=False,*args,**kwargs):
 
    dFD = abs(RM_arr[1]-RM_arr[0])

    # cut out FD range chosen:
    data_use = data[(abs(RM_arr) <= maxFD),:,:]    
    RM_arr_use = RM_arr[(abs(RM_arr) <= maxFD)]  

    # set any data points below PI threshold to NaN:
    data_use[data_use < peakthresh] = np.nan

    moments = {}
    if M0:
        M0_data = dFD*np.nansum(data_use,axis=0)
        M0_data[M0_data == 0] = np.nan
        moments['M0'] = M0_data
    if M1:
        M1_data = dFD*np.nansum(data_use*RM_arr_use[:,np.newaxis,np.newaxis],axis=0)/M0_data
        moments['M1'] = M1_data
    if M2:
        M2_data = np.sqrt(dFD*np.nansum(data_use*(RM_arr_use[:,np.newaxis,np.newaxis]-M1_data)**2,axis=0)/M0_data)
        moments['M2'] = M2_data
        
    return(moments)

def find_pixels(lon,lat,wcs):

    skycoord = SkyCoord(lon, lat, frame=Galactic, unit=(u.deg, u.deg))
                        
    x, y = wcs.world_to_pixel(skycoord)
    
    return x, y

## Get all FD cubes

In [ ]:
hdu_dragons = fits.open('/srv/data/dragons/Mar2024_500_1000_MHz_cube/FD_cube_DRAGONS_CLEAN_32.fits')
hdu_chime   = fits.open('/srv/data/chime/chime_FD_Oct2023_586_729/FDF_clean_tot.fits')
hdu_gmims   = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_FD_PI.fits')

hdr_dragons = hdu_dragons[0].header
hdr_chime   = hdu_chime[0].header
hdr_gmims   = hdu_gmims[0].header

dragons_cube = hdu_dragons[0].data
chime_cube   = hdu_chime[0].data
gmims_cube   = hdu_gmims[0].data

print(dragons_cube.shape)
print(chime_cube.shape)
print(gmims_cube.shape)

FD_dragons = WCS(hdr_dragons).all_pix2world(0,0,range(dragons_cube.shape[0]),0)[2]
FD_chime   = WCS(hdr_chime).all_pix2world(0,0,range(chime_cube.shape[0]),0)[2]
FD_gmims   = WCS(hdr_gmims).all_pix2world(0,0,range(gmims_cube.shape[0]),0)[2]

print(FD_dragons[0],FD_dragons[-1])
print(FD_chime[0],FD_chime[-1])
print(FD_gmims[0],FD_gmims[-1])

## Calculate moments

In [ ]:
moments_dragons = FD_moments(dragons_cube,0,100,FD_dragons,M0=True,M1=True,M2=False)
moments_chime   = FD_moments(chime_cube,0,80,FD_chime,M0=True,M1=True,M2=False)
moments_gmims   = FD_moments(gmims_cube,0.04,450,FD_gmims,M0=True,M1=True,M2=False)

## Make some plots!

In [ ]:
wcs_dragons = WCS(hdr_dragons).dropaxis(2).celestial
wcs_chime   = WCS(hdr_chime).dropaxis(2).celestial
wcs_gmims   = WCS(hdr_gmims).dropaxis(2).celestial

wcs_list = [wcs_dragons, wcs_chime, wcs_gmims]
FD_list = [FD_dragons,FD_chime,FD_gmims]
titles = ['DRAGONS 500-1000 MHz','CHIME 590-730 MHz','GMIMS-HBN 1300-1750 MHz']
cube_list = [dragons_cube, chime_cube, gmims_cube]

In [ ]:
def make_plots(lon_range,lat_range,lon_pt,lat_pt,FD_max=200):

    crange = SkyCoord(lon_range, lat_range, frame=Galactic, unit=(u.deg, u.deg))
    
    fig = plt.figure(figsize=(16,9))
    ax1 = fig.add_subplot(221, projection=wcs_dragons.celestial)
    ax2 = fig.add_subplot(222, projection=wcs_chime.celestial)
    ax3 = fig.add_subplot(223, projection=wcs_gmims.celestial)
    ax4 = fig.add_subplot(224)
    
    im1 = ax1.imshow(moments_dragons['M1'],vmin=-20,vmax=20,cmap='RdBu_r')
    im2 = ax2.imshow(moments_chime['M1'],vmin=-20,vmax=20,cmap='RdBu_r')
    im3 = ax3.imshow(moments_gmims['M1'],vmin=-40,vmax=40,cmap='RdBu_r')

    axs = [ax1,ax2,ax3]
    ims = [im1,im2,im3]
    
    for i in range(0,3):
        axs[i].set_xlabel('Galactic longitude')
        axs[i].set_ylabel('Galactic latitude')
        cbar = fig.colorbar(ims[i], ax=axs[i], shrink=0.8, pad=0.01)
        axs[i].set_title(titles[i]+' M1')
        axs[i].set_ylim(wcs_list[i].world_to_pixel(crange)[1])
        axs[i].set_xlim(wcs_list[i].world_to_pixel(crange)[0])
        x,y = find_pixels(lon_pt,lat_pt,wcs_list[i])
        axs[i].scatter(x,y,s=30,color='k')
        PI = cube_list[i][:,int(np.round(y,0)),int(np.round(x,0))]
        ax4.plot(FD_list[i],PI/np.nanmax(PI),label=titles[i])
    ax4.set_xlim(-FD_max,FD_max)
    ax4.set_ylim(0,1.1)
    ax4.legend(loc='upper right')
    ax4.set_xlabel('Faraday depth (rad/m^2)')
    ax4.set_ylabel('Normalized PI')
    ax4.grid()

    return

In [ ]:
fig = plt.figure(figsize=(16,18))
ax1 = fig.add_subplot(311, projection=wcs_dragons.celestial)
ax2 = fig.add_subplot(312, projection=wcs_chime.celestial)
ax3 = fig.add_subplot(313, projection=wcs_gmims.celestial)

im1 = ax1.imshow(moments_dragons['M1'],vmin=-20,vmax=20,cmap='RdBu_r')
im2 = ax2.imshow(moments_chime['M1'],vmin=-20,vmax=20,cmap='RdBu_r')
im3 = ax3.imshow(moments_gmims['M1'],vmin=-40,vmax=40,cmap='RdBu_r')

axs = [ax1,ax2,ax3]
ims = [im1,im2,im3]

for i in range(0,3):
    axs[i].set_xlabel('Galactic longitude')
    axs[i].set_ylabel('Galactic latitude')
    cbar = fig.colorbar(ims[i], ax=axs[i], shrink=1, pad=0.01)
    axs[i].set_title(titles[i]+' M1')

In [ ]:
lon_range = [180,60]
lat_range = [-10,60]
l = 121
b = 39
make_plots(lon_range,lat_range,l,b,FD_max=150)
#plt.savefig('../PLOTS/gmims_north_l'+str(l)+'_b'+str(b)+'.png')

In [ ]:
lon_range = [180,60]
lat_range = [-10,60]
make_plots(lon_range,lat_range,70,30)

## Tadpole

In [ ]:
lon_range = [180,60]
lat_range = [-10,60]
make_plots(lon_range,lat_range,137,7,FD_max=100)

In [ ]:
lon_range = [180,60]
lat_range = [-10,60]
make_plots(lon_range,lat_range,135,9.5,FD_max=100)